# InferenceGuard: Rewriter Generalization via Teacher Distillation
**Task 1**: Distill complex reasoning from teacher models into a lightweight student model (`Qwen2.5-0.5B-Instruct` / `Qwen2.5-1.5B-Instruct`), persist checkpoints directly to Google Drive, and evaluate generalization against the held-out test split (`data/splits/test.jsonl`).

### Objectives:
1. **Teacher Distillation**: Generate diverse synthetic rewriting demonstrations covering varied tones, complex syntactic clauses, and edge cases with zero bracketed placeholders (`[LOCATION]`, etc.).
2. **Student QLoRA Fine-Tuning**: Fine-tune the small student model with completion-only LM loss masking (`DataCollatorForCompletionOnlyLM`) on ChatML prompts.
3. **Google Drive Fault-Tolerance**: Sync intermediate checkpoints and final model weights directly to mounted Google Drive to survive Colab preemptions/disconnections.
4. **Generalization Validation**: Prove that the distilled student model resolves out-of-distribution generalization failures on `data/splits/test.jsonl` compared to the static prompt baseline (`src/rewriter/prompt_baseline.py`), achieving higher utility preservation and near-zero degeneracy.

## 1. Environment Setup & Branch Checkout
Ensure the repository is cloned and switch to `feature/rewriter-teacher-distillation` containing the distillation dataset, Google Drive callback, and utility evaluation harness.

In [ ]:
# Cell 1: Environment Setup, Branch Checkout & Repository Verification
import os
from pathlib import Path

# Ensure we are in the Inference-Guard directory
if not Path("src/rewriter/train_sft.py").exists():
    if Path("/content/Inference-Guard").exists():
        %cd /content/Inference-Guard
    elif Path("/content/drive/MyDrive/Inference-Guard").exists():
        %cd /content/drive/MyDrive/Inference-Guard
    else:
        %cd /content
        !git clone https://github.com/GIND123/Inference-Guard.git
        %cd Inference-Guard

# Fetch and checkout the distillation feature branch
!git fetch origin feature/rewriter-teacher-distillation
!git checkout -B feature/rewriter-teacher-distillation origin/feature/rewriter-teacher-distillation
!git pull origin feature/rewriter-teacher-distillation

print("\n--- Current Git Status ---")
!git branch -v
!git log -1 --oneline


## 2. Mount Google Drive for Fault-Tolerant Checkpoint Persistence
Mount Google Drive so intermediate training steps and final LoRA adapter weights are saved directly to persistent storage. If your Colab session times out or disconnects, your weights are safely retained.

In [ ]:
# Cell 2: Mount Google Drive (with automatic credential handshake & fallback)
from pathlib import Path

drive_mounted = False
drive_05b_dir = Path("/content/drive/MyDrive/InferenceGuard/rewriter_0.5b")
drive_15b_dir = Path("/content/drive/MyDrive/InferenceGuard/rewriter_1.5b")

try:
    from google.colab import auth, drive
    try:
        # Explicit auth handshake bypasses browser iframe token propagation errors
        auth.authenticate_user()
    except Exception:
        pass

    drive.mount('/content/drive', force_remount=True)
    drive_05b_dir.mkdir(parents=True, exist_ok=True)
    drive_15b_dir.mkdir(parents=True, exist_ok=True)
    drive_mounted = True
    print(f"✓ Google Drive successfully mounted! Checkpoints will persist to:\n - {drive_05b_dir}\n - {drive_15b_dir}")
except Exception as err:
    print("=" * 70)
    print("⚠ Notice: Google Drive mounting could not be completed.")
    print(f"Detail: {err}")
    print("\nCommon fixes if you want Google Drive backup:")
    print(" 1. Multiple Google Accounts: Log into only one account in this browser, or use Incognito.")
    print(" 2. Browser Extensions: Temporarily disable Brave Shields / ad blockers for colab.research.google.com.")
    print(" 3. Left-Panel Icon: Click the 'Files' folder icon on the left sidebar, then click the 'Mount Drive' icon.")
    print("\n→ Proceeding in local storage mode. Checkpoints and weights will save directly to:")
    print("   artifacts/rewriter_qlora_0.5b/ (no work is lost during this active session).")
    print("=" * 70)


## 3. Install Dependencies & Inspect GPU Environment
Install the required training and evaluation libraries (`transformers`, `peft`, `trl`, `datasets`, `bitsandbytes`, `accelerate`).

In [ ]:
# Cell 3: Install dependencies & inspect GPU
# Remove unused multimodal packages that trigger CUDA binary mismatches in Colab
!pip uninstall -y -q torchvision torchaudio
!pip install -q -U transformers peft trl datasets bitsandbytes accelerate

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"BF16 Supported: {torch.cuda.is_bf16_supported()}")


## 4. Dataset Inspection & Teacher Distillation Data Verification
Check that `data/sft/augmented_v1/train.jsonl` and held-out test split `data/splits/test.jsonl` are present.

In [ ]:
# Cell 4: Verify SFT & DPO training data and validate prompt parity
import json
import subprocess
from pathlib import Path

train_path = Path("data/sft/augmented_v1/train.jsonl")
dpo_path = Path("data/dpo/train_dpo.jsonl")

assert train_path.exists(), f"SFT Training dataset missing: {train_path}"
train_count = sum(1 for _ in open(train_path, encoding='utf-8'))
print(f"✓ Verified SFT Dataset: {train_path} ({train_count} balanced samples)")

if dpo_path.exists():
    dpo_count = sum(1 for _ in open(dpo_path, encoding='utf-8'))
    print(f"✓ Verified DPO Preference Dataset: {dpo_path} ({dpo_count} contrastive pairs)")

# Run prompt parity regression gate to ensure zero prompt drift
print("
--- Running Pre-Training Prompt Parity Gate ---")
!python scripts/check_prompt_parity.py data/sft/augmented_v1/train.jsonl

# Preview one sample from SFT training data
with open(train_path, "r", encoding="utf-8") as f:
    sample = json.loads(f.readline())
    print("
Sample SFT Row Structure (messages format served at inference):")
    print(json.dumps(sample, indent=2)[:650] + "
...")


## 5. Fine-Tune Lightweight Student Model (Qwen 0.5B)
Fine-tune `Qwen/Qwen2.5-0.5B-Instruct` using QLoRA with 4-bit quantization and completion-only loss masking.
Intermediate checkpoints (`checkpoint-*`) and final adapter weights will sync directly to Google Drive.

In [ ]:
# Cell 5: Fine-tune Production Student Model via 4-bit QLoRA
# Select model: "Qwen/Qwen2.5-1.5B-Instruct" (production quality ceiling) or "Qwen/Qwen2.5-0.5B-Instruct" (ultra-fast edge)
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT_DIR = "artifacts/rewriter_qlora"

from pathlib import Path
drive_flag = "--drive_path /content/drive/MyDrive/InferenceGuard/rewriter_qlora" if Path("/content/drive/MyDrive").exists() else ""

print(f"Starting QLoRA Fine-Tuning for {MODEL_NAME}...")
print(f"Dataset: data/sft/augmented_v1/train.jsonl (2,000 samples)")
print(f"Epochs: 3 | Effective Batch Size: 16 (4 batch * 4 accum) | LR: 2e-4 with Cosine Decay")

!python -m src.rewriter.train_sft     --model_name {MODEL_NAME}     --output_dir {OUTPUT_DIR}     --dataset_path data/sft/augmented_v1/train.jsonl     --epochs 3     --batch_size 4     --gradient_accumulation_steps 4     --learning_rate 2e-4     --save_steps 50     {drive_flag}

print(f"✓ Training complete! Weights and adapter saved directly to {OUTPUT_DIR}")


## 6. Evaluate 0.5B Student vs Static Prompt Baseline on Held-Out Test Split
Compare the fine-tuned student against the static prompt baseline (`src/rewriter/prompt_baseline.py`) on held-out test data (`data/splits/test.jsonl`).
Computes bidirectional entailment, contradiction probability, token edit ratio, degeneracy rate, and utility preservation.

In [ ]:
# Cell 6: Run utility evaluation on held-out test split
!python -m src.evaluation.utility_metrics \
    --test_split data/splits/test.jsonl \
    --adapter_dir artifacts/rewriter_qlora_0.5b \
    --base_model Qwen/Qwen2.5-0.5B-Instruct \
    --num_samples 50 \
    --compare_baseline \
    --output_report reports/heldout_utility_0.5b.json


In [ ]:
# Cell 7: Display and validate Qwen 0.5B generalization results
import json
from pathlib import Path

report_path = Path("reports/heldout_utility_0.5b.json")
if not report_path.exists():
    raise FileNotFoundError(f"Evaluation report not found at {report_path}. Please check Cell 6 execution logs above.")

with open(report_path, "r", encoding="utf-8") as f:
    report = json.load(f)

print("=" * 75)
print("=== QWEN 0.5B HELD-OUT GENERALIZATION EVALUATION REPORT ===")
print("=" * 75)

baseline = report.get("static_prompt_baseline", {})
student = report.get("teacher_distilled_student", {})
delta = report.get("delta", {})

print(f"{'Metric':<38} | {'Static Baseline':<15} | {'Distilled Student':<17} | {'Delta':<10}")
print("-" * 88)
print(f"{'Bidirectional Entailment':<38} | {baseline.get('mean_bidirectional_entailment', 0):<15.4f} | {student.get('mean_bidirectional_entailment', 0):<17.4f} | {delta.get('bidirectional_entailment_gain', 0):+10.4f}")
print(f"{'Contradiction Prob (lower is better)':<38} | {baseline.get('mean_contradiction_prob', 0):<15.4f} | {student.get('mean_contradiction_prob', 0):<17.4f} | {delta.get('contradiction_drop', 0):+10.4f}")
print(f"{'Utility Preservation Rate':<38} | {baseline.get('utility_preservation_rate', 0):<15.4f} | {student.get('utility_preservation_rate', 0):<17.4f} | {delta.get('utility_preservation_gain', 0):+10.4f}")
print(f"{'Degeneracy Rate (lower is better)':<38} | {baseline.get('degeneracy_rate', 0):<15.4f} | {student.get('degeneracy_rate', 0):<17.4f} | {student.get('degeneracy_rate', 0) - baseline.get('degeneracy_rate', 0):+10.4f}")
print("-" * 88)
print(f"Generalization Out-of-Distribution Improved: {report.get('generalization_improved', False)}")


## 7. (Optional Quality Ceiling) Train & Evaluate Qwen 1.5B Model
Train and evaluate `Qwen2.5-1.5B-Instruct` as a higher-capacity ceiling comparison, saving checkpoints directly to `/content/drive/MyDrive/InferenceGuard/rewriter_1.5b`.

In [ ]:
# Cell 6: Direct Preference Optimization (DPO) Alignment
# Calibrates the fine-tuned student model to penalize over-scrubbing (truncation) and under-scrubbing (PII leaks)
RUN_DPO = True

if RUN_DPO and Path("data/dpo/train_dpo.jsonl").exists():
    print("Starting DPO Alignment on contrastive preference triples...")
    !python src/rewriter/train_dpo.py         --dataset_path data/dpo/train_dpo.jsonl         --model_name_or_path {MODEL_NAME}         --output_dir {OUTPUT_DIR}         --beta 0.1         --epochs 1         --batch_size 2         --grad_accum 8         --lr 5e-6
    print("✓ DPO Alignment successfully applied!")
else:
    print("Skipping DPO phase.")


## 8. Promote Best Adapter to Production & Verify Readiness Gates
Promote the winning student model to `artifacts/rewriter_qlora` and register it with `scripts/check_eval_readiness.py` to satisfy Gate 8 evaluation readiness.

In [ ]:
# Cell 7: Verify Adapter Artifacts and Register in Evaluation Manifest
import json
import shutil
from pathlib import Path

prod_adapter = Path("artifacts/rewriter_qlora")
cfg_file = prod_adapter / "adapter_config.json"

assert cfg_file.exists(), f"adapter_config.json missing in {prod_adapter}! Did training finish?"
print(f"✓ Found valid LoRA adapter in {prod_adapter}")

# Verify readiness gates and register adapter
!python scripts/check_eval_readiness.py --register-adapter


## 9. Interactive Demonstration: Privacy Rewriting in Action
Test the fine-tuned rewriter model live on test prompts with demographic preambles to observe preamble pruning and natural generalization in real-time.

In [ ]:
# Cell 8: Live Inference Demonstration on Challenging Real-World Prompts
import sys
import torch
from pathlib import Path

# Clear cached modules to ensure the newest code is loaded in Colab
for mod in list(sys.modules.keys()):
    if mod.startswith("src.rewriter") or mod.startswith("peft"):
        del sys.modules[mod]

from src.rewriter.inference import QwenRewriterInference

adapter_path = "artifacts/rewriter_qlora"
print(f"Loading Qwen Rewriter from {adapter_path}...")

rewriter = QwenRewriterInference(
    model_name_or_path=MODEL_NAME,
    adapter_path=adapter_path,
    load_in_4bit=torch.cuda.is_available(),
)
print(f"✓ Adapter active: {rewriter.is_adapter_loaded} (Base: {rewriter.resolved_base_model})")

# Test queries covering anaphora, compound requests, embedded PII, and professional credentials
test_prompts = [
    # 1. User's exact MBA admissions query (anaphora + credentials preservation + target university):
    "I am 26 years old living in Denver, CO and working as a software engineer after graduating from Carnegie Mellon. Do you think with these credentials, i can get into an MBA program at harvard?",

    # 2. Compound multi-request with mid-sentence embedded PII:
    "How do I overcome stage fright as a 42 year old software professional, help me find resources for public speaking",

    # 3. Medical consultation with essential clinical invariant:
    "A 72-year-old male with COPD presents with acute dyspnea. I am a 34-year-old internal medicine resident in Chicago. What is the initial ventilatory management?",

    # 4. Legal statutory question with essential jurisdiction:
    "Under California Labor Code Section 2802, are remote employees entitled to reimbursement? FYI I am a 25-year-old software engineer in Mountain View.",
]

print("
" + "=" * 75)
print("=== LIVE INFERENCE DEMONSTRATION ===")
print("=" * 75 + "
")

for i, orig in enumerate(test_prompts, 1):
    out = rewriter.rewrite(orig)
    rewritten = out[0] if isinstance(out, tuple) else out
    engine = "qwen_adapter" if rewriter.is_adapter_loaded and out != rewriter._heuristic_fallback(orig) else "heuristic"
    print(f"[Test Case {i}]")
    print(f"ORIGINAL:  {orig}")
    print(f"REWRITTEN: {rewritten}")
    print(f"ENGINE:    {engine}")
    print("-" * 70)


## 10. Comprehensive 50-Example Benchmark Evaluation
Run the complete 50-prompt test suite (`data/evaluation/benchmark_test_50_annotated.jsonl`) assessing:
1. **Co-occurring PII in Preambles**: Multiple demographic attributes (age, location, occupation, education, salary) clustered at prompt start.
2. **Trailing PII in Postambles**: Personal information introduced after the query (`FYI...`, `By the way...`, parentheticals).
3. **Essential Legal Jurisdictions**: State and local statutes (California, Texas, New York, Florida) that MUST be preserved.
4. **Clinical Patient Presentations**: Third-party patient case studies (infant, child, geriatric parameters) that MUST NOT be scrubbed.
5. **Hybrid Edge Cases**: Legal/medical questions with essential jurisdiction where the user ALSO appends personal demographics.

In [ ]:
# Cell 9: Run Full 53-Case Span-Annotated Benchmark Evaluation
!PYTHONPATH=. python scripts/evaluate_benchmark_50.py     --adapter_path artifacts/rewriter_qlora     --benchmark_file data/evaluation/benchmark_test_50_annotated.jsonl     --output_report reports/benchmark_50_results.json


In [ ]:
# Cell 12: View Full Benchmark Test Cases (Original vs Rewritten without Truncation)
import json
from pathlib import Path

report_file = Path("reports/benchmark_50_results.json")
if report_file.exists():
    with open(report_file, "r", encoding="utf-8") as f:
        data = json.load(f)
    print(f"=== BENCHMARK REPORT (Total: {data.get('total_cases')}, Passed: {data.get('passed_cases')} [{data.get('pass_rate')*100:.1f}%]) ===\n")
    for tc in data.get("results", []):
        status = "[PASS]" if tc.get("passed") else "[FAIL]"
        print(f"{status} #{tc.get('id')} ({tc.get('category')}):")
        print(f"    ORIGINAL:  {tc.get('input')}")
        print(f"    REWRITTEN: {tc.get('rewritten')}")
        if tc.get("notes"):
            print(f"    ISSUES:    {', '.join(tc.get('notes'))}")
        print("-" * 70)
else:
    print("No benchmark report found. Run Cell 11 first!")


## 11. Launch InferenceGuard Interactive Web UI
Launch the FastAPI + Uvicorn server in the background and expose the web dashboard via Google Colab port forwarding or a public tunnel.


In [ ]:
# Cell 10: Launch InferenceGuard Interactive Web UI (Uvicorn + Colab iframe / Tunnel)
!pip install -q fastapi uvicorn nest-asyncio
!pkill -f uvicorn 2>/dev/null || true
!fuser -k 8000/tcp 2>/dev/null || true

import os
import sys
import time
import shutil
import subprocess
import urllib.request
import re
from pathlib import Path

# 1. Ensure production adapter is present in artifacts/rewriter_qlora
src_dpo = Path("artifacts/rewriter_dpo_0.5b")
src_05b = Path("artifacts/rewriter_qlora_0.5b")
prod_adapter = Path("artifacts/rewriter_qlora")

if not (prod_adapter / "adapter_config.json").exists():
    source = None
    if (src_dpo / "adapter_config.json").exists():
        source = src_dpo
    elif (src_05b / "adapter_config.json").exists():
        source = src_05b
    if source:
        prod_adapter.mkdir(parents=True, exist_ok=True)
        for f in source.glob("*"):
            if f.is_file():
                shutil.copy2(f, prod_adapter / f.name)
        print(f"Copied student adapter from {source} -> {prod_adapter}")

# 2. Pre-flight check: Verify backend engine imports cleanly
print("Verifying backend engine imports...")
try:
    import web.api
    print("[OK] Backend modules verified.")
except Exception as e:
    print(f"[ERROR] Import failure: {e}")
    raise

# 3. Launch Uvicorn in background, redirecting output to uvicorn.log
log_path = Path("uvicorn.log")
log_file = open(log_path, "w", encoding="utf-8")

print("Starting Uvicorn backend server on port 8000...")
proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "web.api:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=log_file,
    stderr=subprocess.STDOUT,
)

# 4. Active Health Check Polling (waits until port is active and healthy)
server_ready = False
for attempt in range(45):
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=1) as resp:
            if resp.status == 200:
                server_ready = True
                print(f"[SUCCESS] InferenceGuard server is LIVE and HEALTHY (ready in {attempt + 1}s)!\n")
                break
    except Exception:
        pass
    time.sleep(1)
    if (attempt + 1) % 5 == 0:
        print(f"  Initializing neural models... ({attempt + 1}s elapsed)")

if not server_ready:
    print("\n[WARNING] Server did not respond within 45s. Recent logs:")
    log_file.flush()
    if log_path.exists():
        print(log_path.read_text(encoding="utf-8")[-2000:])
else:
    # 5. Access Method 1: Cloudflare Quick Tunnel (Free, zero accounts, works in new tabs)
    if sys.platform.startswith("linux"):
        cf_bin = Path("cloudflared")
        if not cf_bin.exists():
            try:
                urllib.request.urlretrieve(
                    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
                    str(cf_bin),
                )
                cf_bin.chmod(0o755)
            except Exception:
                pass
        if cf_bin.exists():
            try:
                cf_proc = subprocess.Popen(
                    ["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
                    stderr=subprocess.PIPE,
                    stdout=subprocess.DEVNULL,
                    text=True,
                )
                t_end = time.time() + 12
                cf_url = None
                while time.time() < t_end and cf_proc.poll() is None:
                    line = cf_proc.stderr.readline() if cf_proc.stderr else ""
                    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
                    if m:
                        cf_url = m.group(0)
                        break
                    time.sleep(0.1)
                if cf_url:
                    print("=" * 70)
                    print("PUBLIC TAB ACCESS LINK (No account or cookies required):")
                    print(f"  {cf_url}")
                    print("=" * 70 + "\n")
            except Exception as e:
                print(f"Cloudflare tunnel notice: {e}")

    # 6. Access Method 2: Direct In-Notebook Embedded UI
    if "google.colab" in sys.modules:
        try:
            from google.colab.output import serve_kernel_port_as_iframe
            print("=" * 70)
            print("IN-NOTEBOOK INTERACTIVE INTERFACE (Embedded directly below):")
            print("=" * 70)
            serve_kernel_port_as_iframe(8000, height=850)
        except Exception as e:
            print(f"iframe note: {e}")